# 4. Обучение классификаторов

**Внимание:** все обученные веса лежат в `models/*.pth`
Этот ноутбук можно не запускать — сразу переходите к `05_evaluation.ipynb`

Здесь приведён код обучения для воспроизводимости:
- InceptionV1 + GELU (лучшая модель, 90/10)
- ResNet18 + WRS + аугментации
- InceptionV3
- ResNet50

## Гиперпараметры
- Optimizer: Adam, lr=1e-4
- Batch size: 32
- Image size: 256×256
- Loss: CrossEntropyLoss

In [ ]:
import sys
sys.path.append("..")

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler, random_split
from torchvision.datasets import ImageFolder
from torchvision.transforms import transforms
from collections import Counter

from src.config import (
    DATA_PROCESSED_90_10, MODELS_DIR, FIGURES_DIR, DEVICE,
    NORM_MEAN, NORM_STD, BATCH_SIZE, IMG_SIZE, LEARNING_RATE, NUM_EPOCHS, SEED,
)
from src.models import InceptionV1, InceptionV3, ResNet50
from src.train import train_classifier
from src.utils import set_seed, plot_history

set_seed(SEED)

TRAIN_DIR = DATA_PROCESSED_90_10 / "train_dataset"
TEST_DIR  = DATA_PROCESSED_90_10 / "test_dataset"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

train_ds = ImageFolder(str(TRAIN_DIR), transform=train_transform)
full_test_ds = ImageFolder(str(TEST_DIR), transform=test_transform)


generator = torch.Generator().manual_seed(SEED)
val_ds, test_ds = random_split(full_test_ds, [0.5, 0.5], generator=generator)

# WeightedRandomSampler против дисбаланса
labels = train_ds.targets
class_counts = Counter(labels)
weights = [1.0 / class_counts[l] for l in labels]
sampler = WeightedRandomSampler(weights, num_samples=len(labels), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE,
                          num_workers=4, pin_memory=True)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
print(f"Class counts: {dict(class_counts)}")

In [ ]:
MODEL_FACTORY = {
    "inception_v1": (InceptionV1, {"in_channels": 3, "num_classes": 2}, "inception_v1.pth"),
    "inception_v3": (InceptionV3, {"in_channels": 3, "num_classes": 2}, "inception_v3.pth"),
    "resnet50":     (ResNet50,     {},                                 "resnet50.pth"),
}

for name, (cls, kw, _) in MODEL_FACTORY.items():
    m = cls(**kw)
    n_params = sum(p.numel() for p in m.parameters())
    print(f"{name:14s} | params = {n_params:,}")
    del m

In [ ]:


for name, (cls, kw, weight_file) in MODEL_FACTORY.items():
    print(f"Обучаем: {name}\n")

    model = cls(**kw).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    save_path = str(MODELS_DIR / weight_file)

    history = train_classifier(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        num_epochs=EPOCHS,
        optimizer=optimizer,
        loss_function=criterion,
        save_name=save_path,
    )

    plot_history(
        history_path=str(MODELS_DIR / weight_file).replace(".pth", ".history.json"),
        save_path=str(FIGURES_DIR / f"{name}_history.png"),
    )

    print(f"{name} готов. Веса: {save_path}")
